# Session 27 · Hybrid search and reranking
**Homework notebook.** In class you watched meaning search miss `SKU-EL-004`, keyword search miss "the parcel arrived smashed",
and hybrid search plus a reranker get both. Here you run every step yourself, then build the fusion and the reranking step
in four TODOs in `s27/hybrid.py`.

| Part | What | Time |
|---|---|---|
| A | Why one kind of search is never enough, how BM25 works, how to merge two rankings, what a reranker is | 20 min reading |
| B | Run it: keyword → vectors → hybrid → the RRF arithmetic → rerank → the full comparison → the gate → hybrid inside Qdrant | 40 min |
| C | Your homework file: four TODOs, run, check, commit | 40 min |
| D | Interview questions and what comes next | 10 min |

**Before you start** (course repo folder, PowerShell):
```powershell
git pull
uv sync                                                  # NEW: rank-bm25 (keyword search, pure Python)
uv run python module04/s27_demo.py download              # the reranker model (~80 MB) - once, on good Wi-Fi
uv run jupyter lab module04/s27_hybrid_rerank.ipynb
```
**This is also Gate 1 practice:** the checkpoint after this session covers RAG fundamentals (S24–S27). If you can explain every
cell of Part B in your own words, you're ready.

---
# Part A — Why one search is never enough

### The problem in two questions
| Customer types | Meaning search (vectors, S24) | Keyword search |
|---|---|---|
| "the parcel arrived **smashed**" | ✅ finds *Damaged, defective or wrong items* — it knows smashed ≈ damaged | ❌ no shared word |
| "**SKU-EL-004**" | ❌ a product code has no "meaning"; the vector is a vague guess | ✅ exact match, top result |
| "**WELCOME10**", "**HDFC** EMI", "**RuPay**", "**26cm** pan" | ⚠️ often misses — rare names and numbers | ✅ |

Real customers type **both kinds** — often in the same chat. Order ids, SKUs, coupon codes, bank names, sizes and pincodes are
exactly the words a support assistant must not miss. So production systems run **both searches and merge them**. That's
**hybrid search**. ShopAssist Pro does it in `app/rag/index.py`; Weaviate, Qdrant, Elasticsearch, OpenSearch, Azure AI Search and
Pinecone all offer it built in.

### How this session is built
```
question ──► BM25 (keywords) ──► top 20 ids ─┐
         └─► vectors (meaning) ─► top 20 ids ─┴─► RRF fusion ─► top 20 ──► reranker ──► top 3 ──► gate ──► LLM
                                                  (Stage 1: cheap, wide)       (Stage 2: slow, precise)   ("I don't know"?)
```
Two corpora this time: the **help centre** (53 sections, S24) and the **product catalog** (30 products from Mini-Project 1; doc_id = SKU).
22 test questions: 13 **meaning** questions (customers' own words) and 9 **exact** questions (codes, names, numbers).

### BM25 — "Ctrl+F with a score"
BM25 is the keyword-scoring formula behind Elasticsearch, OpenSearch, Lucene and Solr — the default in search engines for 25 years.
It scores a document for a question by adding up a share for every question word the document contains. Three ideas:

| Idea | Why | In the formula |
|---|---|---|
| **Rare words count more** | "HDFC" appears in one section → very informative. "order" appears everywhere → tells you nothing | **IDF** (inverse document frequency) |
| **Repeats help, but less each time** | A section saying "refund" 5 times is a bit more about refunds than one saying it once — not 5× more | **k1** (default 1.5): term frequency *saturates* |
| **Long documents don't win just by being long** | A long section contains more words by accident | **b** (default 0.75): length normalisation |

$$\text{BM25}(q, d) = \sum_{w \in q} \text{IDF}(w) \cdot \frac{tf(w,d) \cdot (k_1 + 1)}{tf(w,d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{avgdl})}$$

You don't memorise it — you remember the three ideas. BM25 is a **sparse vector** (S24's "word checklist"): one slot per word,
almost all zeros. Before scoring, text is **tokenised**: lower case, stop words (the, is, my…) removed, plurals trimmed
(shoes → shoe), and codes like `sku-fw-001` kept whole. Tokenisation decides what can ever match.

### Merging two rankings — RRF
BM25 scores run 0 to 20+; cosine runs 0 to 1. **Adding them is meaningless** — BM25 would drown the vectors. Two fixes:

| Method | How | Used by |
|---|---|---|
| **RRF** (Reciprocal Rank Fusion) | Ignore scores; use **ranks**. Each list gives a row `1 / (60 + rank)`; add them up | Qdrant, Elasticsearch, OpenSearch, Azure AI Search, ShopAssist Pro |
| **Weighted score fusion** | Min-max each list to 0..1, then `alpha × vector + (1 − alpha) × keyword` | Weaviate (`alpha`), Pinecone |

Why 60? It's the constant from the original 2009 paper; it keeps one first place from dominating. Rank 1 gives 1/61 ≈ 0.0164,
rank 10 gives 1/70 ≈ 0.0143 — so **a row near the top of both lists beats a row that is first in only one**.

### Reranking — shortlist, then interview
Our embedder is a **bi-encoder**: question and document are turned into vectors **separately**. That's why documents can be embedded
once, in advance, and search is fast — but the model never reads the question *next to* the passage.

A **cross-encoder** takes question + passage **together** and outputs one relevance score. Much more accurate (it sees how every
question word relates to every passage word), but it must run once **per pair at question time** — far too slow for a million documents.

So production RAG uses **two stages**, like hiring: **shortlist 20 CVs quickly** (hybrid search), then **interview those 20 properly**
(the reranker), and keep the best 3. Today's reranker: `Xenova/ms-marco-MiniLM-L-6-v2` (~80 MB, CPU, free). Managed alternatives:
Cohere Rerank, Jina Reranker, Voyage rerank, Vertex AI ranking; bigger open models: `BAAI/bge-reranker-v2-m3`.

### The relevance gate
Every method returns *something* — even for "do you sell used cars". If we pass that passage to the LLM, it answers from it.
A **gate** drops weak results so the LLM gets **no passage rather than a wrong one** and says "I don't know". Gate on a
reranker score, a cosine threshold, or keyword coverage (ShopAssist Pro uses keyword coverage OR cosine ≥ 0.63).

### The words you need (full list: `module04/terms_s27.md`)
| Term | Plain meaning |
|---|---|
| **Hybrid search** | Run keyword search and vector search, merge the two result lists |
| **BM25** | The standard keyword-scoring formula: rare words count more, repeats saturate, long documents normalised |
| **Tokenisation** | Turning text into the words a search engine compares (lower case, stop words, stemming, codes) |
| **IDF** | Inverse document frequency — how rare a word is across all documents |
| **Sparse vs dense** | One slot per word, mostly zeros (BM25) vs every number used (embeddings) |
| **Fusion** | Merging several ranked lists into one |
| **RRF** | Reciprocal Rank Fusion: add `1/(60 + rank)` from each list |
| **alpha** | In weighted fusion, the share given to the vector score (1 = vectors only) |
| **Bi-encoder** | Embeds question and document separately (fast, pre-computable) |
| **Cross-encoder / reranker** | Reads question + document together and scores the pair (slow, accurate) |
| **Two-stage retrieval** | Cheap wide recall (top 20), then precise reranking (top 3) |
| **Relevance gate** | A threshold below which we return nothing, so the LLM can say "I don't know" |

---
# Part B — Run it
### B1. Load both corpora and the test questions
The first run embeds both corpora with bge-small (S24's model) and caches the vectors in `module04/.s27_cache/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "module04") if (Path.cwd() / "module04").exists() else str(Path.cwd()))
from dotenv import find_dotenv, load_dotenv; load_dotenv(find_dotenv(usecwd=True))

import s27_utils as S27
import s27_queries as Q

rs = {c: S27.Retriever(c) for c in S27.CORPORA}
for c, r in rs.items():
    print(f"{c:<9} {len(r.chunks)} documents, e.g. {r.chunks[0].source}  [{r.chunks[0].doc_id}]")
print(len(Q.CASES), "test questions:", sum(k == "exact" for *_, k in Q.CASES), "exact,",
      sum(k == "meaning" for *_, k in Q.CASES), "meaning")

### B2. Tokenisation — what BM25 actually compares
Notice: stop words gone, `shoes` → `shoe`, and the product code kept as one token **as well as** its parts.

In [ ]:
for text in ["What is the price of SKU-FW-002 shoes?", "HDFC no-cost EMI", "the parcel arrived smashed"]:
    print(f"{text!r:<44} -> {S27.tokenize(text)}")

### B3. Keyword vs vectors on the two kinds of question
Run both cells. Write down which method found the right document each time.

In [ ]:
def show(hits):
    for h in hits:
        print(f"   {h.score:8.3f}  {h.chunk.source}  [{h.chunk.doc_id}]")

for question, corpus in [("SKU-EL-004", "products"), ("the parcel arrived smashed", "help")]:
    r = rs[corpus]
    print(f'\n"{question}"  ({corpus})')
    print(" keyword:"); show(r.keyword(question))
    print(" vectors:"); show(r.dense(question))

### B4. Where a BM25 score comes from
Each line is one question word: how rare it is (idf), how often it appears in this section, and what it added.

In [ ]:
r = rs["help"]
q = "HDFC no-cost EMI"
for h in r.keyword(q, 2):
    print(f"{h.score:6.2f}  {h.chunk.source}")
    for w, idf, tf, part in r.bm25.explain(q, r.keys.index(h.key)):
        if tf:
            print(f"        {w:<10} idf {idf:4.2f}  x{tf}  -> +{part:.2f}")

### B5. RRF by hand
Both lists give their top 20. Look at the table: which rows are near the top of **both** lists?

In [ ]:
r, q = rs["products"], "PowerCell"
kw, dn = r.keyword_ranking(q, 20), r.dense_ranking(q, 20)
print(f"{'row':<22}{'keyword rank':>13}{'vector rank':>12}{'RRF':>9}")
for key, s in S27.rrf([kw, dn])[:6]:
    a = kw.index(key) + 1 if key in kw else "-"
    b = dn.index(key) + 1 if key in dn else "-"
    print(f"{key:<22}{a!s:>13}{b!s:>12}{s:>9.4f}")

### B6. Rerank: hybrid top 20 → cross-encoder → top 3
The reranker's scores are **logits**: above 0 roughly means "relevant", far below 0 means "unrelated". Compare the order with
plain hybrid.

In [ ]:
for question, corpus in [("can I pay the courier in cash when it arrives", "help"), ("is the Stride available in UK 9", "products")]:
    r = rs[corpus]
    print(f'\n"{question}"')
    print(" hybrid:"); show(r.hybrid(question))
    print(" rerank:"); show(r.rerank(question))

### B7. The full comparison — 22 questions × 5 methods
This is the table from class, on your laptop. Expected story: **keyword wins exact, vectors win meaning, hybrid is good at both,
rerank lifts MRR** (the right answer moves to position 1). If your numbers tell a different story, look at the misses — that's
how real teams tune retrieval.

In [ ]:
print(f"{'method':<10}{'hit@3':>8}{'MRR':>7}{'exact':>8}{'meaning':>9}{'ms/q':>7}")
results = {}
for m in (*S27.METHODS, "weighted"):
    res = results[m] = S27.evaluate(rs, Q.CASES, m)
    a, e, n = res["all"], res["exact"], res["meaning"]
    print(f"{m:<10}{a['hits']:>5}/{a['n']:<2}{a['mrr']:>7.2f}{e['hits']:>5}/{e['n']:<2}{n['hits']:>6}/{n['n']:<2}{res['ms']:>7.0f}")

print("\nmisses of 'hybrid':")
for row in results["hybrid"]["rows"]:
    if row["rank"] is None:
        print(f"  {row['question']!r} wanted {row['expected']}, got {row['found']}")

**Try it:** change `alpha` in weighted fusion and watch exact vs meaning trade off:

In [ ]:
for alpha in (0.0, 0.3, 0.5, 0.7, 1.0):
    class W:                                   # a tiny wrapper so evaluate() calls weighted() with this alpha
        def __init__(self, r): self.r = r
        def search(self, m, q, k=3): return self.r.weighted(q, k, alpha=alpha)
    res = S27.evaluate({c: W(r) for c, r in rs.items()}, Q.CASES, "weighted")
    print(f"alpha {alpha:.1f}:  exact {res['exact']['hits']}/{res['exact']['n']}   meaning {res['meaning']['hits']}/{res['meaning']['n']}")

### B8. The relevance gate
Look at the top reranker score for in-scope and out-of-scope questions. Where would you draw the line? You'll choose your
number in TODO 4.

In [ ]:
r = rs["help"]
for q, label in [(q, "in") for q, c, *_ in Q.CASES if c == "help"][:6] + [(q, "OUT") for q, _ in Q.OUT_OF_SCOPE]:
    print(f"{r.rerank(q, k=1)[0].score:7.2f}  {label:<4} {q}")

### B9. The same hybrid search inside Qdrant
Qdrant stores **two vectors per point**: the dense one (384 numbers) and a **sparse** BM25 one (a slot per word). One
`query_points` call runs both searches (**prefetch**) and fuses them with RRF on the server. Uses the S26 Docker server if it's
running, otherwise in-process Qdrant.

In [ ]:
import s26_utils as S26
client, where = S26.connect_qdrant("auto")
name = S27.qdrant_hybrid_load(client, rs["products"])
print("loaded", name, "->", where)
for score, p in S27.qdrant_hybrid_search(client, rs["products"], "SKU-EL-004"):
    print(f"   {score:.3f}  {p['source']}  [{p['doc_id']}]")

---
# Part C — Your homework: `s27/hybrid.py`
| TODO | What |
|---|---|
| 1 | `rrf_fuse(rankings, k=60)` — Reciprocal Rank Fusion |
| 2 | `weighted_fuse(keyword, dense, alpha)` — min-max score fusion, the Weaviate way |
| 3 | `rerank_top(question, candidates, score_fn, k)` — stage 2: score all 20 in one batch, keep the best k |
| 4 | Three exact-term questions of your own, and your relevance-gate threshold with a reason |

Your file plugs YOUR functions into the course retriever, runs all 22 questions plus your 3 with five methods, tests your gate,
and writes `results.md`.

```powershell
uv run python module04/s27_setup.py                   # copies the file to <your work repo>\s27\hybrid.py
uv run python "<your work repo>\s27\hybrid.py"        # YOUR fusion, YOUR reranking step; writes results.md
uv run pytest tests/test_s27.py -rs                   # the checker (offline)
cd "<your work repo>" ; git add s27 ; git commit -m "S27: hybrid search" ; git push
```

---
# Part D — Interview questions
| Question | Fresher | Experienced |
|---|---|---|
| Why hybrid search? | Vectors miss exact words like codes; keywords miss meaning. Using both covers both. | Dense retrieval underperforms on rare tokens (SKUs, ids, names, numbers) and out-of-domain vocabulary; BM25 fails on paraphrase. Hybrid with RRF is a robust default; measure per question type (exact vs semantic) on a golden set. |
| Explain BM25. | Keyword scoring where rare words count more. | Sum over query terms of IDF × saturated TF with length normalisation; k1 controls saturation, b controls length penalty; quality depends on tokenisation (stemming, stop words, code handling). |
| RRF vs weighted fusion? | RRF uses ranks, weighted uses scores. | RRF is scale-free and needs no tuning (k = 60); weighted fusion needs score normalisation and an alpha tuned per dataset, but can encode a preference. Min-max is sensitive to outliers. |
| Bi-encoder vs cross-encoder? | One embeds separately, the other reads both together. | Bi-encoder: independent embeddings, pre-computable, ANN-searchable, lower accuracy. Cross-encoder: joint attention over the pair, higher accuracy, O(candidates) model calls per query — so rerank only the top 20–100. |
| How many candidates do you rerank? | About 20. | Trade recall vs latency: measure recall@N of stage 1 and p95 latency of stage 2; typical 20–100; batch the pairs; cache by query. |
| How does RAG say "I don't know"? | Use a threshold. | A calibrated relevance gate (reranker score, cosine or keyword coverage) tuned on in-scope and out-of-scope questions; return no context and an explicit refusal path; track the rejection rate in eval. |

### What's next
**S28 — Consolidation and Gate 1.** Everything from S24–S27 in one practical: embed, store, search hybrid, rerank, evaluate.
Then Module 5 builds full RAG on top of this retriever.